In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
from scipy.signal import butter, filtfilt, find_peaks, hilbert
import warnings
import re
fs = 200  # fréquence d'échantillonnage (Hz)

In [8]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ====== PARAMETRES ======
dossier = "Coda_data"
fs = 200  # fréquence d'échantillonnage (Hz)

fichiers_S01 = [
    "Gr4_S01_L1_20250011_001.txt",
    "Gr4_S01_L1_20250011_002.txt",
    "Gr4_S01_L1_20250011_003.txt",
    "Gr4_S01_L1_20250011_004.txt",
    "Gr4_S01_L2_20250011_005.txt",
    "Gr4_S01_L2_20250011_006.txt",
    "Gr4_S01_L2_20250011_007.txt",
    "Gr4_S01_L2_20250011_008.txt",
    "Gr4_S01_L3_20250011_009.txt",
    "Gr4_S01_L3_20250011_010.txt",
    "Gr4_S01_L3_20250011_011.txt",
    "Gr4_S01_L3_20250011_012.txt",
    "Gr4_S01_L4_20250011_013.txt",
    "Gr4_S01_L4_20250011_014.txt",
    "Gr4_S01_L4_20250011_015.txt",
    "Gr4_S01_L4_20250011_016.txt",
]

# ====== STOCKAGE ======
resultats = []
resultats_angle = []

# ====== BOUCLE FICHIERS ======
for f in fichiers_S01:
    chemin = os.path.join(dossier, f)

    df = pd.read_csv(chemin, sep="\t", header=2)
    df.columns = df.columns.str.strip()
    df = df.replace("NaN", np.nan)

    # nettoyage
    colonnes_a_supprimer = [c for c in df.columns if ".V" in c or c.strip() == ""]
    df = df.drop(columns=colonnes_a_supprimer, errors="ignore")

    time_col = "Time" if "Time" in df.columns else "Time (s)"

    # garder markers 5-8
    pattern = r"Marker 0[5-8]\.[XYZ]"
    colonnes_markers = [c for c in df.columns if re.search(pattern, c)]
    df = df[[time_col] + colonnes_markers]
    df = df.apply(pd.to_numeric, errors="coerce")

    # ====== CALCULS ======
    cx = ((df["Marker 05.X"] + df["Marker 06.X"]) / 2 +
          (df["Marker 07.X"] + df["Marker 08.X"]) / 2) / 2

    cy = ((df["Marker 05.Y"] + df["Marker 06.Y"]) / 2 +
          (df["Marker 07.Y"] + df["Marker 08.Y"]) / 2) / 2

    cz = ((df["Marker 05.Z"] + df["Marker 06.Z"]) / 2 +
          (df["Marker 07.Z"] + df["Marker 08.Z"]) / 2) / 2

    top_x = (df["Marker 05.X"] + df["Marker 06.X"]) / 2
    top_y = (df["Marker 05.Y"] + df["Marker 06.Y"]) / 2

    # ====== INTERPOLATION ======
    for signal in [cx, cy, cz, top_x, top_y]:
        nan_idx = np.isnan(signal)
        if np.any(nan_idx):
            signal[nan_idx] = np.interp(
                np.flatnonzero(nan_idx),
                np.flatnonzero(~nan_idx),
                signal[~nan_idx]
            )

    # ====== NOM COLONNES ======
    m = re.search(r"Gr4_(S\d+)_(L\d+)_\d+_(\d+)\.txt$", f)
    if m:
        col_base = f"{m.group(1)}_{m.group(2)}_{int(m.group(3))}"
    else:
        col_base = f.replace(".txt", "")

    # ====== DATAFRAME POSITION ======
    df_res = pd.DataFrame({
        "Time": df[time_col],
        f"{col_base}_X": cx,
        f"{col_base}_Y": cy,
        f"{col_base}_Z": cz
    })

    resultats.append(df_res)

    # ====== DATAFRAME ANGLE ======
    df_angle = pd.DataFrame({
        "Time": df[time_col],
        f"{col_base}_center_X": cx,
        f"{col_base}_center_Y": cy,
        f"{col_base}_top_X": top_x,
        f"{col_base}_top_Y": top_y
    })

    resultats_angle.append(df_angle)

# ====== FUSION ======
df_final = resultats[0]
for df in resultats[1:]:
    df_final = df_final.merge(df, on="Time")

df_angle_base = resultats_angle[0]
for df in resultats_angle[1:]:
    df_angle_base = df_angle_base.merge(df, on="Time")

print("✅ Fusion terminée")

# ====== DOSSIERS ======
dossier_root = "Cinematique_S01_L"
os.makedirs(dossier_root, exist_ok=True)

dossier_time = os.path.join(dossier_root, "plots_time_RAW")
dossier_fft = os.path.join(dossier_root, "plots_FFT")
os.makedirs(dossier_time, exist_ok=True)
os.makedirs(dossier_fft, exist_ok=True)

# ====== PLOTS ======
time_col = "Time"
cols = [c for c in df_final.columns if c != time_col]

for col in cols:

    signal = df_final[col].values
    time = df_final[time_col].values

    # ====== PLOT TEMPOREL ======
    plt.figure(figsize=(8,3))
    plt.plot(time, signal)

    plt.xlabel("Temps (s)")
    plt.ylabel(col)
    plt.title(f"{col} RAW")

    plt.xticks(np.arange(0, float(time[-1]) + 1, 5))
    plt.tight_layout()

    plt.savefig(os.path.join(dossier_time, f"{col}.png"), dpi=150)
    plt.close()

    # ====== FFT ======
    signal_centered = signal - np.mean(signal)  # enlever offset
    N = len(signal_centered)

    fft_vals = np.fft.fft(signal_centered)
    fft_vals = np.abs(fft_vals) / N

    freqs = np.fft.fftfreq(N, d=1/fs)

    mask = freqs >= 0
    freqs = freqs[mask]
    fft_vals = fft_vals[mask]

    # ====== PLOT FFT ======
    plt.figure(figsize=(8,3))
    plt.plot(freqs, fft_vals)

    plt.xlabel("Fréquence (Hz)")
    plt.ylabel("Amplitude")
    plt.title(f"{col} - FFT")

    plt.xlim(0, 20)# zoom utile biomécanique
    plt.ylim(0,30)
    plt.tight_layout()

    plt.savefig(os.path.join(dossier_fft, f"{col}_FFT.png"), dpi=150)
    plt.close()

print("✅ Tous les plots générés")
print(f"📁 Time : {dossier_time}")
print(f"📁 FFT  : {dossier_fft}")

✅ Fusion terminée
✅ Tous les plots générés
📁 Time : Cinematique_S01_L\plots_time_RAW
📁 FFT  : Cinematique_S01_L\plots_FFT
